# B5: Granger Causality & Temporal Motifs

Validates RFC-007 features on two domains:

| Feature | Dataset | Question |
|---------|---------|----------|
| `granger_causality` | Trump tweets + S&P 500 | Does presidential rhetoric predict market movements? |
| `granger_causality` | eRisk | Does community-level language shift precede individual deterioration? |
| `discover_motifs` | eRisk | Do depressed users show recurring behavioral patterns? |
| `discover_discords` | eRisk | Can we detect the most anomalous behavioral period? |

In [1]:
import chronos_vector as cvx
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import json, time, os, warnings
warnings.filterwarnings('ignore')

DATA_DIR = '../data'
EMB_DIR = f'{DATA_DIR}/embeddings'

C_DEP = '#e74c3c'
C_CTL = '#3498db'
C_ACCENT = '#2ecc71'
C_MARKET = '#f39c12'

---
## Part A: Granger Causality — Trump Tweets → Market Movements

**Hypothesis**: Trump's tweet rhetoric (projected to anchors like 'trade_war', 'economy', 'federal_reserve') Granger-causes S&P 500 sector movements with a 1-5 day lag.

In [2]:
# Load Trump tweet embeddings and market data from B3
# These were generated in the B3_trump_impact notebook
TRUMP_CACHE = f'{DATA_DIR}/cache/trump_anchor_daily.parquet'
MARKET_CACHE = f'{DATA_DIR}/cache/market_anchor_daily.parquet'

if os.path.exists(TRUMP_CACHE) and os.path.exists(MARKET_CACHE):
    trump_df = pd.read_parquet(TRUMP_CACHE)
    market_df = pd.read_parquet(MARKET_CACHE)
    print(f'Trump daily: {len(trump_df)} days')
    print(f'Market daily: {len(market_df)} days')
else:
    print('⚠️ B3 cached data not found. Run B3_trump_impact notebook first.')
    print('Generating synthetic proxy for demonstration...')
    
    # Synthetic: Trump tone → market reaction with 2-day lag
    np.random.seed(42)
    n_days = 500
    K = 5  # anchor dimensions
    lag = 2
    
    trump_signal = np.cumsum(np.random.randn(n_days, K) * 0.05, axis=0)
    market_signal = np.zeros((n_days, K))
    for i in range(lag, n_days):
        market_signal[i] = 0.6 * trump_signal[i - lag] + np.random.randn(K) * 0.1
    
    base_ts = 1_500_000_000_000_000  # ~2017
    day_us = 86_400_000_000
    
    trump_traj = [(base_ts + i * day_us, trump_signal[i].tolist()) for i in range(n_days)]
    market_traj = [(base_ts + i * day_us, market_signal[i].tolist()) for i in range(n_days)]
    print(f'Synthetic: {n_days} days, K={K} anchors, planted lag={lag} days')

⚠️ B3 cached data not found. Run B3_trump_impact notebook first.
Generating synthetic proxy for demonstration...
Synthetic: 500 days, K=5 anchors, planted lag=2 days


In [3]:
# Build trajectories for Granger test
if 'trump_traj' not in dir():
    # From real data: extract anchor columns as trajectory
    anchor_cols = [c for c in trump_df.columns if c.startswith('anchor_')]
    K = len(anchor_cols)
    trump_traj = [(int(row['timestamp'].value), row[anchor_cols].values.astype(float).tolist())
                  for _, row in trump_df.iterrows()]
    market_traj = [(int(row['timestamp'].value), row[anchor_cols].values.astype(float).tolist())
                   for _, row in market_df.iterrows()]

print(f'Trump trajectory: {len(trump_traj)} points, K={K}')
print(f'Market trajectory: {len(market_traj)} points')

Trump trajectory: 500 points, K=5
Market trajectory: 500 points


In [4]:
# Granger causality test: Trump → Market
result_t2m = cvx.granger_causality(trump_traj, market_traj, max_lag=5, significance=0.05)

print('=== Trump → Market ===')
print(f'  Direction:    {result_t2m["direction"]}')
print(f'  Optimal lag:  {result_t2m["optimal_lag"]} days')
print(f'  F-statistic:  {result_t2m["f_statistic"]:.4f}')
print(f'  p-value:      {result_t2m["p_value"]:.6f}')
print(f'  Effect size:  {result_t2m["effect_size"]:.4f}')

=== Trump → Market ===
  Direction:    bidirectional
  Optimal lag:  1 days
  F-statistic:  398.5419
  p-value:      0.000000
  Effect size:  0.4433


In [5]:
# Test multiple lag values to find the optimal
lag_results = []
for max_lag in range(1, 11):
    r = cvx.granger_causality(trump_traj, market_traj, max_lag=max_lag, significance=0.10)
    lag_results.append({
        'max_lag': max_lag,
        'direction': r['direction'],
        'optimal_lag': r['optimal_lag'],
        'f_stat': r['f_statistic'],
        'p_value': r['p_value'],
        'effect_size': r['effect_size'],
    })

lag_df = pd.DataFrame(lag_results)
print(lag_df.to_string(index=False))

 max_lag     direction  optimal_lag     f_stat       p_value  effect_size
       1 bidirectional            1 398.541859 4.386192e-190     0.443318
       2 bidirectional            1 398.541859 4.386192e-190     0.443318
       3 bidirectional            1 398.541859 4.386192e-190     0.443318
       4 bidirectional            1 398.541859 4.386192e-190     0.443318
       5 bidirectional            1 398.541859 4.386192e-190     0.443318
       6 bidirectional            1 398.541859 4.386192e-190     0.443318
       7 bidirectional            1 398.541859 4.386192e-190     0.443318
       8 bidirectional            1 398.541859 4.386192e-190     0.443318
       9 bidirectional            1 398.541859 4.386192e-190     0.443318
      10 bidirectional            1 398.541859 4.386192e-190     0.443318


In [6]:
# Visualize Granger lag analysis
fig = make_subplots(rows=1, cols=2, subplot_titles=['F-statistic by Max Lag', 'p-value by Max Lag'])
fig.add_trace(go.Bar(x=lag_df['max_lag'], y=lag_df['f_stat'],
                     marker_color=C_MARKET, name='F-stat'), row=1, col=1)
fig.add_trace(go.Scatter(x=lag_df['max_lag'], y=lag_df['p_value'],
                         mode='lines+markers', marker_color=C_DEP, name='p-value'), row=1, col=2)
fig.add_hline(y=0.05, line_dash='dash', line_color='gray', row=1, col=2,
              annotation_text='α=0.05')
fig.update_layout(height=400, title='Granger Causality: Trump Rhetoric → Market Impact')
fig.show()

---
## Part B: Temporal Motifs & Discords on eRisk

**Hypothesis**: Depressed users show recurring semantic patterns (weekly/monthly cycles) and more pronounced discords (anomalous behavioral periods) than controls.

In [7]:
# Load eRisk data
df_full = pd.read_parquet(f'{EMB_DIR}/erisk_mental_embeddings.parquet')
emb_cols = [c for c in df_full.columns if c.startswith('emb_')]

dep_users = df_full[df_full['label'] == 'depression']['user_id'].unique()
ctl_users = df_full[df_full['label'] == 'control']['user_id'].unique()
np.random.seed(42)
ctl_sample = np.random.choice(ctl_users, size=min(len(dep_users), len(ctl_users)), replace=False)

# Load anchor vectors
ANCHOR_CACHE = f'{DATA_DIR}/cache/dsm5_anchors.npz'
data = np.load(ANCHOR_CACHE, allow_pickle=True)
anchor_vectors = data['anchor_vectors'].item()
healthy_vector = data['healthy_vector']
anchors_np = np.array([anchor_vectors[k] for k in anchor_vectors.keys()] + [healthy_vector])
anchor_names = list(anchor_vectors.keys()) + ['healthy']
K = len(anchor_names)

print(f'Loaded eRisk: dep={len(dep_users)}, ctl={len(ctl_sample)}, K={K} anchors')

Loaded eRisk: dep=233, ctl=233, K=10 anchors


In [8]:
# Build anchor-projected trajectories
def build_anchor_traj(user_ids):
    results = {}
    for uid in user_ids:
        grp = df_full[df_full['user_id'] == uid].sort_values('timestamp')
        if len(grp) < 20:  # need enough points for motif detection
            continue
        traj = [(int(row['timestamp'].value), row[emb_cols].values.astype(np.float32).tolist())
                for _, row in grp.iterrows()]
        projected = cvx.project_to_anchors(traj, anchors_np.tolist(), 'cosine')
        results[uid] = projected
    return results

dep_anchor_trajs = build_anchor_traj(dep_users)
ctl_anchor_trajs = build_anchor_traj(ctl_sample)
print(f'Dep trajectories (≥20 pts): {len(dep_anchor_trajs)}')
print(f'Ctl trajectories (≥20 pts): {len(ctl_anchor_trajs)}')

Dep trajectories (≥20 pts): 213
Ctl trajectories (≥20 pts): 217


In [9]:
# Discover motifs for each group
WINDOW = 7  # 7-post window (roughly weekly for active users)

def analyze_motifs(user_trajs, label):
    stats = []
    for uid, traj in user_trajs.items():
        if len(traj) < 2 * WINDOW:
            continue
        try:
            motifs = cvx.discover_motifs(traj, WINDOW, max_motifs=3, exclusion_zone=0.5)
            discords = cvx.discover_discords(traj, WINDOW, max_discords=3)
            
            best_motif = motifs[0] if motifs else None
            best_discord = discords[0] if discords else None
            
            stats.append({
                'user_id': uid, 'label': label,
                'n_motifs': len(motifs),
                'best_motif_distance': best_motif['mean_match_distance'] if best_motif else None,
                'best_motif_occurrences': len(best_motif['occurrences']) if best_motif else 0,
                'best_motif_period': best_motif['period'] if best_motif else None,
                'best_discord_distance': best_discord[2] if best_discord else None,
                'traj_len': len(traj),
            })
        except Exception:
            pass
    return pd.DataFrame(stats)

dep_motif_stats = analyze_motifs(dep_anchor_trajs, 'depression')
ctl_motif_stats = analyze_motifs(ctl_anchor_trajs, 'control')

print(f'Depression: {len(dep_motif_stats)} users analyzed')
print(f'Control:    {len(ctl_motif_stats)} users analyzed')

Depression: 213 users analyzed
Control:    217 users analyzed


In [10]:
# Compare motif characteristics
all_stats = pd.concat([dep_motif_stats, ctl_motif_stats])

print('=== MOTIF COMPARISON ===')
for col in ['best_motif_distance', 'best_motif_occurrences', 'best_discord_distance']:
    dep_mean = dep_motif_stats[col].mean()
    ctl_mean = ctl_motif_stats[col].mean()
    print(f'{col}:')
    print(f'  Depression: {dep_mean:.4f}')
    print(f'  Control:    {ctl_mean:.4f}')
    print()

=== MOTIF COMPARISON ===
best_motif_distance:
  Depression: 0.0234
  Control:    0.0205

best_motif_occurrences:
  Depression: 32.9577
  Control:    20.5438

best_discord_distance:
  Depression: 0.0618
  Control:    0.0747



In [11]:
# Visualize discord severity distribution
fig = go.Figure()
fig.add_trace(go.Box(y=dep_motif_stats['best_discord_distance'].dropna(),
                     name='Depression', marker_color=C_DEP))
fig.add_trace(go.Box(y=ctl_motif_stats['best_discord_distance'].dropna(),
                     name='Control', marker_color=C_CTL))
fig.update_layout(
    title='Top Discord Severity (NN Distance) by Group',
    yaxis_title='Nearest-Neighbor Distance (higher = more anomalous)',
    height=400,
)
fig.show()

In [12]:
# Period detection: do depressed users show more regular patterns?
dep_periods = dep_motif_stats['best_motif_period'].dropna()
ctl_periods = ctl_motif_stats['best_motif_period'].dropna()

print(f'Period detected in {len(dep_periods)}/{len(dep_motif_stats)} depressed users '
      f'({100*len(dep_periods)/max(len(dep_motif_stats),1):.0f}%)')
print(f'Period detected in {len(ctl_periods)}/{len(ctl_motif_stats)} control users '
      f'({100*len(ctl_periods)/max(len(ctl_motif_stats),1):.0f}%)')
if len(dep_periods) > 0:
    print(f'Depression mean period: {dep_periods.mean():.1f} steps')
if len(ctl_periods) > 0:
    print(f'Control mean period:    {ctl_periods.mean():.1f} steps')

Period detected in 2/213 depressed users (1%)
Period detected in 4/217 control users (2%)
Depression mean period: 6.5 steps
Control mean period:    29.8 steps


---
## Summary

In [13]:
print('=== B5 VALIDATION RESULTS ===')
print()
print('Granger Causality (Trump → Market):')
print(f'  Direction: {result_t2m["direction"]}')
print(f'  Lag: {result_t2m["optimal_lag"]} days, p={result_t2m["p_value"]:.6f}')
print()
print('Motif Analysis (eRisk):')
print(f'  Dep mean motif distance: {dep_motif_stats["best_motif_distance"].mean():.4f}')
print(f'  Ctl mean motif distance: {ctl_motif_stats["best_motif_distance"].mean():.4f}')
print(f'  Dep mean discord severity: {dep_motif_stats["best_discord_distance"].mean():.4f}')
print(f'  Ctl mean discord severity: {ctl_motif_stats["best_discord_distance"].mean():.4f}')

=== B5 VALIDATION RESULTS ===

Granger Causality (Trump → Market):
  Direction: bidirectional
  Lag: 1 days, p=0.000000

Motif Analysis (eRisk):
  Dep mean motif distance: 0.0234
  Ctl mean motif distance: 0.0205
  Dep mean discord severity: 0.0618
  Ctl mean discord severity: 0.0747
